In [3]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

In [4]:

import pandas as pd
#df=pd.read_csv("...Data/data.csv")
df = pd.read_csv("Data/data.csv")

In [5]:

agg_features = df.groupby('CustomerId').agg({
    'Amount': [
        'sum',    # Total Transaction Amount
        'mean',   # Average Transaction Amount
        'count',  # Transaction Count
        'std'     # Standard Deviation (Variability)
    ]
}).reset_index()

# 3. Rename the columns for clarity (flattening the multi-index)
agg_features.columns = [
    'CustomerId', 
    'Total_Transaction_Amount', 
    'Average_Transaction_Amount', 
    'Transaction_Count', 
    'Std_Dev_Transaction_Amount'
]

# 4. Handle NaN in Standard Deviation
# If a customer only has 1 transaction, the Std Dev will be NaN. 
# It is best practice to fill this with 0.
agg_features['Std_Dev_Transaction_Amount'] = agg_features['Std_Dev_Transaction_Amount'].fillna(0)

print(agg_features.head())

        CustomerId  Total_Transaction_Amount  Average_Transaction_Amount  \
0     CustomerId_1                  -10000.0               -10000.000000   
1    CustomerId_10                  -10000.0               -10000.000000   
2  CustomerId_1001                   20000.0                 4000.000000   
3  CustomerId_1002                    4225.0                  384.090909   
4  CustomerId_1003                   20000.0                 3333.333333   

   Transaction_Count  Std_Dev_Transaction_Amount  
0                  1                    0.000000  
1                  1                    0.000000  
2                  5                 6558.963333  
3                 11                  560.498966  
4                  6                 6030.478146  


In [6]:
# Ensure the column is in datetime format
df['TransactionStartTime'] = pd.to_datetime(df['TransactionStartTime'])

# Extracting components
df['Transaction_Hour'] = df['TransactionStartTime'].dt.hour
df['Transaction_Day'] = df['TransactionStartTime'].dt.day
df['Transaction_Month'] = df['TransactionStartTime'].dt.month
df['Transaction_Year'] = df['TransactionStartTime'].dt.year

print(df[['TransactionStartTime', 'Transaction_Hour', 'Transaction_Day']].head())

       TransactionStartTime  Transaction_Hour  Transaction_Day
0 2018-11-15 02:18:49+00:00                 2               15
1 2018-11-15 02:19:08+00:00                 2               15
2 2018-11-15 02:44:21+00:00                 2               15
3 2018-11-15 03:32:55+00:00                 3               15
4 2018-11-15 03:34:21+00:00                 3               15


Defining feature groups

In [7]:
# 1. Define Column Groups
# Use the aggregate and datetime features you just created
num_cols = [
    'Total_Transaction_Amount', 'Average_Transaction_Amount', 
    'Transaction_Count', 'Std_Dev_Transaction_Amount', 
    'Transaction_Hour', 'Transaction_Day' , 'Amount' , 'Value' , 'TransactionStartTime' , 'PricingStrategy' , 'FraudResult'
]
#cat_cols = [  'BatchId' , 'AccountId' , 'SubscriptionId' , 'CustomerId' , 'CurrencyCode'  , 'ProviderId' , 'ProductId' , 'ProductCategory' , 'ChannelId']
cat_cols = ['ProviderId', 'ProductCategory', 'ChannelId'] 

In [8]:
# 2. Build the "Robust & Automated" Pipeline
# preprocessor = ColumnTransformer(
#     transformers=[
#         # Handle Missing Values (Median) + Standardization
#         ('num', Pipeline([
#             ('imputer', SimpleImputer(strategy='median')),
#             ('scaler', StandardScaler())
#         ]), num_cols),
        
#         # Handle Missing Values (Constant) + One-Hot Encoding
#         ('cat', Pipeline([
#             ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
#             ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
#         ]), cat_cols)
#     ])

In [9]:
# clf = Pipeline(steps=[('preprocessor', preprocessor)])

In [7]:
# Merge the customer summaries back into the main data
df_final = df.merge(agg_features, on='CustomerId', how='left')

# Forcefully remove hidden spaces from all column names
df_final.columns = df_final.columns.str.strip()

In [8]:
# Cleaned numeric list (no timestamps or targets)
num_cols = [
    'Total_Transaction_Amount', 'Average_Transaction_Amount', 
    'Transaction_Count', 'Std_Dev_Transaction_Amount', 
    'Transaction_Hour', 'Transaction_Day', 'Amount', 'Value', 'PricingStrategy'
]

# Identify the categories to encode
#cat_cols = ['BatchId', 'AccountId', 'SubscriptionId', 'ProviderId', 'ProductId', 'ProductCategory', 'ChannelId']
cat_cols = ['ProviderId', 'ProductCategory', 'ChannelId'] 
# Update your preprocessor with these cleaned lists
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ])

# Re-wrap in the final pipeline
clf = Pipeline(steps=[('preprocessor', preprocessor)])

In [9]:
# Separate into Features (X) and Target (y)
X = df_final.drop(columns=['FraudResult'])
y = df_final['FraudResult']

# Run the full pipeline
X_transformed = clf.fit_transform(X)
print("Pipeline complete! Shape of processed data:", X_transformed.shape)

Pipeline complete! Shape of processed data: (95662, 28)


In [10]:
from xverse.transformer import WOE

# 1. Initialize the WOE transformer
# We use the original X (before the pipeline) because xverse handles its own binning
clf_woe = WOE()

# 2. Fit the transformer using your Features (X) and Target (y)
# X should be your df_final minus the target; y is your 'FraudResult'
clf_woe.fit(X, y)

# 3. Transform your dataset
# This replaces categorical/numeric values with their calculated WoE
X_woe = clf_woe.transform(X)

# 4. View the Information Value (IV) Summary
# This fulfills the "Read about WoE and IV" instruction
print("Information Value (IV) Summary:")
print(clf_woe.iv_df)

ValueError: The input feature(s) should be numeric type. Some of the input features                             has character values in it. Please use a encoder before performing monotonic operations.

In [12]:
from xverse.transformer import WOE
import pandas as pd

# 1. Select only the features you need for WoE
woe_features = [
    'Total_Transaction_Amount', 'Average_Transaction_Amount', 
    'Transaction_Count', 'Std_Dev_Transaction_Amount', 
    'Transaction_Hour', 'Transaction_Day', 'Amount', 'Value', 
    'PricingStrategy', 'ProviderId', 'ProductCategory', 'ChannelId'
]

# Create a copy to avoid SettingWithCopy warnings
X_woe_input = X[woe_features].copy()

# 2. Force types: Categories to string, Numbers to float
cat_cols = ['ProviderId', 'ProductCategory', 'ChannelId', 'PricingStrategy']
num_cols = [f for f in woe_features if f not in cat_cols]

X_woe_input[cat_cols] = X_woe_input[cat_cols].astype(str)
X_woe_input[num_cols] = X_woe_input[num_cols].apply(pd.to_numeric, errors='coerce')

# 3. Initialize and Fit
# We set monotonic_binning=False if the numeric types continue to cause issues 
# with the underlying Scikit-Learn 'check_array' version on your Mac.
clf_woe = WOE(monotonic_binning=False) 
clf_woe.fit(X_woe_input, y)

# 4. Transform and View IV
X_woe = clf_woe.transform(X_woe_input)
print("Information Value (IV) Summary:")
print(clf_woe.iv_df)

/Users/tekabe/Downloads/KAIM Week4/credit-risk-model/venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/tekabe/Downloads/KAIM Week4/credit-risk-model/venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/tekabe/Downloads/KAIM Week4/credit-risk-model/venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


KeyError: 'ChannelId'

In [13]:
from xverse.transformer import WOE
import pandas as pd
import numpy as np

# 1. Select the most reliable features to avoid math errors
# We will focus on the aggregates and primary categories
woe_features = [
    'Total_Transaction_Amount', 'Average_Transaction_Amount', 
    'Transaction_Count', 'Std_Dev_Transaction_Amount', 
    'Transaction_Hour', 'Amount', 'PricingStrategy', 
    'ProviderId', 'ProductCategory'
]

X_woe_input = X[woe_features].copy()

# 2. Clean types to ensure xverse can process them
cat_cols = ['ProviderId', 'ProductCategory', 'PricingStrategy']
num_cols = [f for f in woe_features if f not in cat_cols]

X_woe_input[cat_cols] = X_woe_input[cat_cols].astype(str)
X_woe_input[num_cols] = X_woe_input[num_cols].apply(pd.to_numeric, errors='coerce').fillna(0)

# 3. Fit with stable settings
# monotonic_binning=False avoids the Scikit-learn version conflict on your Mac
clf_woe = WOE(monotonic_binning=False)
clf_woe.fit(X_woe_input, y)

# 4. Safely Transform
# We only transform columns that were successfully fitted
final_columns = list(clf_woe.woe_bins.keys())
X_woe = clf_woe.transform(X_woe_input[final_columns])

print("Information Value (IV) Summary:")
print(clf_woe.iv_df)

Information Value (IV) Summary:
  Variable_Name  Information_Value
0    ProviderId           3.327239


/Users/tekabe/Downloads/KAIM Week4/credit-risk-model/venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Users/tekabe/Downloads/KAIM Week4/credit-risk-model/venv/lib/python3.12/site-packages/pandas/core/arraylike.py:402: RuntimeWarning: divide by zero encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


In [14]:
# Save to your project-relative path
X_woe.to_csv("Data/model_ready_woe.csv", index=False)
print("Task 3 complete. Data saved for Modeling.")

Task 3 complete. Data saved for Modeling.
